# TIM x Sapienza Machine Unlearning Hackathon

Notebook di analisi e presentazione. Il workflow di submission e' autonomo e si
esegue con `python main.py`; qui ispezioniamo dati, baseline e configurazioni
senza ridefinire la logica di produzione.

## 1. Obiettivo e criteri

Vogliamo rimuovere l'influenza degli utenti forget preservando la Precision@10 e
contenendo il tempo. La MIA ufficiale e' nascosta: ogni segnale privacy locale e'
una proxy diagnostica, non il punteggio della challenge.

## 2. Setup e riproducibilita'

Importiamo soltanto API del package. Il seed governa split, inizializzazione e
DataLoader; nessuna cella dipende da stato creato in un kernel precedente.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "machine_unlearning").is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from machine_unlearning.data import load_challenge_data, validate_data_model_compatibility
from machine_unlearning.metrics import evaluate_model
from machine_unlearning.model import build_model_from_artifact, load_model_artifact
from machine_unlearning.submission import validate_submission

SEED = 92
VALIDATION_FRACTION = 0.11
DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
SUBMISSION_DIR = REPO_ROOT / "submission"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 3. Dati e split

Carichiamo gli shard nell'ordine dei file, verifichiamo schema e ID, escludiamo
prima il forget set e dividiamo il retain in training e validation. La validation
misura utility, ma non e' automaticamente un vero non-member set del modello
originale.

In [ ]:
data = load_challenge_data(
    DATA_DIR,
    validation_fraction=VALIDATION_FRACTION,
    seed=SEED,
)

split_summary = pd.DataFrame(
    {
        "split": ["training completo", "retain", "retain train", "validation", "forget"],
        "righe": [
            len(data.full_frame),
            len(data.retain_frame),
            len(data.retain_train_frame),
            len(data.validation_frame),
            len(data.forget_frame),
        ],
    }
)
display(split_summary)
display(pd.Series(data.replaced_feature_values, name="valori feature sostituiti"))


## 4. Preprocessing e modello originale

Le feature non numeriche o non finite diventano zero, come nel codice originale;
le target invalide causano un errore. Ricostruiamo l'architettura dell'artifact e
carichiamo parametri e buffer con `strict=True`.

In [ ]:
artifact = load_model_artifact(DATA_DIR / "model_artifact")
validate_data_model_compatibility(data, artifact["architecture"])
original_model = build_model_from_artifact(artifact, device=DEVICE)

{
    "device": str(DEVICE),
    "feature": len(data.schema.feature_columns),
    "target": len(data.schema.target_columns),
    "architettura": artifact["architecture"],
}


## 5. Baseline di utility

Precision@10 e BCE sui logit hanno una sola implementazione nel package. La
selezione finale non puo' dipendere soltanto dalla Precision@10: un modello utile
puo' conservare ancora il comportamento membership che vogliamo rimuovere.

In [ ]:
original_validation = evaluate_model(
    original_model,
    data.x_validation,
    data.y_validation,
    device=DEVICE,
)
original_forget = evaluate_model(
    original_model,
    data.x_forget,
    data.y_forget,
    device=DEVICE,
)

pd.DataFrame(
    [
        {
            "modello": "originale",
            "validation_precision_at_10": original_validation["precision_at_10"],
            "validation_bce": original_validation["bce_from_logits"],
            "forget_bce": original_forget["bce_from_logits"],
        }
    ]
)


## 6. Baseline e configurazioni

Il retraining da zero e' il riferimento metodologico piu' pulito, perche' non
vede gli utenti forget. Lo script separato confronta retraining e candidati
ibridi; non implementa una baseline autonoma di retain fine-tuning. La
configurazione canonica riporta nel proprio selection_note l'evidenza locale
che ne ha motivato la promozione e i limiti ancora aperti.

In [ ]:
final_config_path = REPO_ROOT / "configs/final_config.json"
search_config_path = REPO_ROOT / "configs/search_configs.json"
final_config = json.loads(final_config_path.read_text(encoding="utf-8"))
search_config = json.loads(search_config_path.read_text(encoding="utf-8"))

display(pd.Series(final_config, name="configurazione finale"))
print(f"Configurazioni ibride di base: {len(search_config['candidates'])}")
print(f"Stato: {final_config.get('selection_status', 'non dichiarato')}")
print(final_config.get('selection_note', 'Nessuna nota di selezione.'))


## 7. Metodo ibrido esplorato

1. **Fisher diagonale**: media dei gradienti per-esempio al quadrato, con modello
   in `eval()` per non aggiornare BatchNorm.
2. **Fisher ratio**: confronta importanza forget e retain; un gate assoluto evita
   ratio grandi causati da denominatori quasi nulli.
3. **Selective Fisher Dampening**: attenua solo i pesi selezionati. I buffer
   BatchNorm non sono parametri e non entrano nella maschera.
4. **Selective gradient ascent**: opzionale e mascherato; aumentare soltanto la
   loss forget puo' produrre instabilita' e perdita di utility.
5. **Repair e distillation**: BCE retain, MSE tra logit teacher/student e
   regolarizzazione parametrica. Distilliamo i logit per conservare informazione
   funzionale piu' ricca delle sole label binarie.
6. **Ricalibrazione BatchNorm**: usa esclusivamente dati retain.

## 8. Privacy proxy e ricerca

La proxy distingue originale e retrained sugli stessi utenti forget usando fold
raggruppati per utente. Serve alla selezione locale insieme a utility e tempo;
non e' la MIA ufficiale, non prova la cancellazione esatta e non garantisce la
leaderboard. Gli output sono ignorati da Git e compaiono solo dopo una ricerca
reale; questo notebook non incorpora metriche simulate.

In [ ]:
search_output_dir = OUTPUT_DIR / "search"
comparison_path = search_output_dir / "search_comparison.csv"
finalists_path = search_output_dir / "finalists.csv"
proposed_config_path = search_output_dir / "proposed_final_config.json"
metadata_path = search_output_dir / "search_metadata.json"

available_outputs = pd.Series(
    {
        "search_comparison.csv": comparison_path.is_file(),
        "finalists.csv": finalists_path.is_file(),
        "proposed_final_config.json": proposed_config_path.is_file(),
        "search_metadata.json": metadata_path.is_file(),
    },
    name="presente",
)
display(available_outputs)

if comparison_path.is_file():
    display(pd.read_csv(comparison_path).head(20))
if finalists_path.is_file():
    display(pd.read_csv(finalists_path))
if proposed_config_path.is_file():
    proposed_config = json.loads(proposed_config_path.read_text(encoding="utf-8"))
    display(pd.Series(proposed_config, name="configurazione proposta"))
if metadata_path.is_file():
    search_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    display(pd.json_normalize(search_metadata, sep=".").T)

if not available_outputs.any():
    print(
        "Nessuna evidenza completa in outputs/search. "
        "Eseguire la ricerca descritta nel README; la modalita' quick usa "
        "outputs/search/quick e non promuove la configurazione canonica."
    )


## 9. Submission e validazione

`main.py` produce soltanto `model_artifact`, `execution_time.txt` e
`validation_ids.csv`. Il validator ricostruisce lo split, controlla artifact e
ID ed esegue inferenza su un piccolo batch reale.

In [ ]:
if SUBMISSION_DIR.is_dir():
    validation_report = validate_submission(SUBMISSION_DIR, data_dir=DATA_DIR)
    display(pd.Series(validation_report, name="submission"))
else:
    print("Submission non ancora generata. Eseguire: python main.py")


## 10. Conclusioni

Il notebook racconta gli esperimenti, ma non e' una dipendenza operativa. La
configurazione canonica documenta una promozione basata su risultati reali locali.
Le metriche locali verificano coerenza, utility e stabilita' limitata;
soltanto l'evaluator ufficiale puo' misurare la MIA nascosta e la leaderboard.